<a href="https://colab.research.google.com/github/HrishikeshWadile/SoftwarePractice/blob/main/Reinforced_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gym torch numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

try:
    import gymnasium as gym
except:
    import gym

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
env = gym.make("CartPole-v1")

# Handle reset for all gym versions
reset_result = env.reset()
if isinstance(reset_result, tuple):
    state = reset_result[0]
else:
    state = reset_result

state_dim = len(state)
action_dim = env.action_space.n

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
policy_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [ ]:
memory = deque(maxlen=10000)
batch_size = 64
gamma = 0.99

In [ ]:
episodes = 200
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01

In [ ]:
for episode in range(episodes):

    reset_result = env.reset()
    if isinstance(reset_result, tuple):
        state = reset_result[0]
    else:
        state = reset_result

    state = torch.FloatTensor(state)
    total_reward = 0
    done = False

    while not done:

        # Epsilon-Greedy Action
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                q_values = policy_net(state)
                action = torch.argmax(q_values).item()

        # Step (handles all gym versions)
        step_result = env.step(action)

        if len(step_result) == 5:
            next_state, reward, terminated, truncated, _ = step_result
            done = terminated or truncated
        else:
            next_state, reward, done, _ = step_result

        next_state_tensor = torch.FloatTensor(next_state)

        # Store in memory
        memory.append((state, action, reward, next_state_tensor, done))

        state = next_state_tensor
        total_reward += reward

        # Train if enough samples
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)

            states, actions, rewards, next_states, dones = zip(*batch)

            states = torch.stack(states)
            actions = torch.tensor(actions)
            rewards = torch.tensor(rewards, dtype=torch.float32)
            next_states = torch.stack(next_states)
            dones = torch.tensor([float(d) for d in dones])

            # Current Q values
            current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()

            # Target Q values
            next_q = target_net(next_states).max(1)[0]
            target_q = rewards + gamma * next_q * (1 - dones)

            loss = criterion(current_q, target_q.detach())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Update target network
    if episode % 10 == 0:
        target_net.load_state_dict(policy_net.state_dict())

    print(f"Episode {episode+1}, Reward: {total_reward}")

env.close()

Episode 1, Reward: 16.0
Episode 2, Reward: 14.0
Episode 3, Reward: 31.0
Episode 4, Reward: 20.0
Episode 5, Reward: 63.0
Episode 6, Reward: 17.0
Episode 7, Reward: 32.0
Episode 8, Reward: 15.0
Episode 9, Reward: 19.0
Episode 10, Reward: 23.0
Episode 11, Reward: 18.0
Episode 12, Reward: 11.0
Episode 13, Reward: 16.0
Episode 14, Reward: 14.0
Episode 15, Reward: 23.0
Episode 16, Reward: 15.0
Episode 17, Reward: 15.0
Episode 18, Reward: 28.0
Episode 19, Reward: 48.0
Episode 20, Reward: 23.0
Episode 21, Reward: 17.0
Episode 22, Reward: 13.0
Episode 23, Reward: 23.0
Episode 24, Reward: 36.0
Episode 25, Reward: 13.0
Episode 26, Reward: 22.0
Episode 27, Reward: 25.0
Episode 28, Reward: 27.0
Episode 29, Reward: 20.0
Episode 30, Reward: 19.0
Episode 31, Reward: 14.0
Episode 32, Reward: 19.0
Episode 33, Reward: 23.0
Episode 34, Reward: 17.0
Episode 35, Reward: 15.0
Episode 36, Reward: 14.0
Episode 37, Reward: 13.0
Episode 38, Reward: 25.0
Episode 39, Reward: 25.0
Episode 40, Reward: 13.0
Episode 4